In [17]:
from pathlib import Path
import os
import scipy.io
import numpy as np
import mne

# Portable dataset configuration.
# Option 1: place KARA ONE in <repository>/data/KARA_ONE.
# Option 2: set the KARA_ONE_DATASET_DIR environment variable.
PROJECT_ROOT = Path.cwd().resolve()
DATASET_ROOT = Path(
    os.environ.get(
        "KARA_ONE_DATASET_DIR",
        PROJECT_ROOT / "data" / "KARA_ONE",
    )
).expanduser().resolve()

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"KARA ONE dataset directory not found: {DATASET_ROOT}\n"
        "Place the dataset in data/KARA_ONE or set "
        "KARA_ONE_DATASET_DIR to its location."
    )

print(f"Using dataset: {DATASET_ROOT}")


In [18]:
# Audit the dataset
files = list(DATASET_ROOT.rglob("*.mat"))
print(f"Found {len(files)} MAT files total.\n")

for file in files:
    # 1. Skip the summary feature/index files
    if any(x in file.name for x in ['features', 'inds', 'regression', 'results']):
        continue

    try:
        mat_data = scipy.io.loadmat(file)
        
        # Kara ONE subject files usually contain a 'data' structure
        if 'data' in mat_data:
            struct = mat_data['data']
            
            # Extract fields from the MATLAB structure array
            # MATLAB structures loaded via scipy require [0, 0] indexing
            try:
                # X contains the signal: shape is usually (channels, samples) or (trials, channels, samples)
                eeg_data = struct[0, 0]['X']  
                sampling_rate = float(struct[0, 0]['sr'][0, 0])
                
                shape = eeg_data.shape
                
                # Check if data is 3D (Trials x Channels x Samples) or 2D (Channels x Samples)
                if len(shape) == 3:
                    num_trials, num_channels, num_samples = shape
                    total_samples = num_trials * num_samples
                    data_type = f"Epoched ({num_trials} trials)"
                else:
                    num_channels = min(shape)
                    total_samples = max(shape)
                    data_type = "Continuous Raw"

                duration_seconds = total_samples / sampling_rate

                print(f"Subject File: {file.name}")
                print(f"  Type:          {data_type}")
                print(f"  Channels:      {num_channels}")
                print(f"  Sampling rate: {sampling_rate} Hz")
                print(f"  Total Duration:{duration_seconds:.2f} seconds")
                print("-" * 40)
                
            except (IndexError, KeyError, ValueError) as e:
                print(f"Structure mismatch in {file.name}: {e}")
        else:
            # Document files that don't fit the expected subject data layout
            print(f"Non-subject data file found: {file.name} (Keys: {list(mat_data.keys())})")

    except Exception as e:
        print(f"Error reading {file.name}: {e}")

Found 85 MAT files total.

Non-subject data file found: epoch_data.mat (Keys: ['__header__', '__version__', '__globals__', 'epoch_data'])


In [19]:
#Find and inspect epoch_data.mat
print("--- SCANNING FOR 'epoch_data.mat' ---")
epoch_files = list(DATASET_ROOT.rglob("epoch_data.mat"))

if not epoch_files:
    print("Could not find epoch_data.mat anywhere inside the directory structure.")
else:
    print(f"Total 'epoch_data.mat' files found: {len(epoch_files)}")
    print("\n--- FOUND PATHS ---")
    for f in epoch_files:
        print(f)
    
    # Use the first match found
    file_path = epoch_files[0]
    print(f"\n=== INSPECTING: {file_path.name} ===")
    
    mat_data = scipy.io.loadmat(file_path)
    core_structure = mat_data['epoch_data'][0, 0]
    
    # Grab the imagined speech data array
    thinking_trials = core_structure['thinking_mats']
    
    # Peak into the very first trial (Index 0, 0)
    first_trial = thinking_trials[0, 0]
    
    print(f"Trial object type: {type(first_trial)}")
    print(f"Trial matrix shape: {first_trial.shape}")
    print("-" * 30)
    
    # Deduce data formatting
    shape = first_trial.shape
    num_channels = min(shape)
    num_samples = max(shape)
    
    print(f"Each trial contains:")
    print(f"  - Channels: {num_channels}")
    print(f"  - Time Samples: {num_samples}")

--- SCANNING FOR 'epoch_data.mat' ---
Total 'epoch_data.mat' files found: 1

--- FOUND PATHS ---
/Users/jessicathiessen/Documents/Cognitive Systems/neurotech/KARA_datatset/p 4/spoclab/users/szhao/EEG/data/MM08/epoch_data.mat

=== INSPECTING: epoch_data.mat ===
Trial object type: <class 'numpy.ndarray'>
Trial matrix shape: (64, 5000)
------------------------------
Each trial contains:
  - Channels: 64
  - Time Samples: 5000


In [20]:
#Unpack MATLAB structure details
print("=== UNPACKING MATLAB STRUCTURE ===")

# 1. Extract the core object out of the (1,1) wrapper
raw_element = mat_data['epoch_data']
core_structure = raw_element[0, 0]
fields = core_structure.dtype.names

print(f"Available fields inside 'epoch_data': {fields}\n")

# 2. Dynamically look at the shape of each field inside the structure
for field_name in fields:
    field_content = core_structure[field_name]
    print(f"Field: '{field_name}'")
    print(f"  - Type:  {type(field_content)}")
    print(f"  - Shape: {field_content.shape}")
    
    # Peek at the first element if it's small, or give a description
    if field_content.size == 1:
        print(f"  - Value: {field_content[0,0]}")
    print("-" * 30)

# Extract and display main data matrix info
eeg_matrix = mat_data['epoch_data']
shape = eeg_matrix.shape

print("\n=== KARA ONE DATA STRUCTURE ===")
print(f"Matrix type: {type(eeg_matrix)}")
print(f"Matrix shape: {eeg_matrix.shape}")
print(f"Number of dimensions: {len(eeg_matrix.shape)}")

if len(shape) == 3:
    print("\nInterpretation (3D Epoch Matrix):")
    print(f"  - Dimension 0 (Size {shape[0]}): Likely the total number of Trials/Epochs.")
    print(f"  - Dimension 1 (Size {shape[1]}): Likely the EEG Channels.")
    print(f"  - Dimension 2 (Size {shape[2]}): The number of Time-series samples per trial.")
elif len(shape) == 2:
    print("\nInterpretation (2D Matrix):")
    print(f"  - Axis 0 (Size {shape[0]}): Check if this matches your total channel count or total samples.")
    print(f"  - Axis 1 (Size {shape[1]}): The corresponding pairs.")

=== UNPACKING MATLAB STRUCTURE ===
Available fields inside 'epoch_data': ('thinking_mats', 'clearing_mats', 'stimuli_mats', 'speaking_mats')

Field: 'thinking_mats'
  - Type:  <class 'numpy.ndarray'>
  - Shape: (1, 131)
------------------------------
Field: 'clearing_mats'
  - Type:  <class 'numpy.ndarray'>
  - Shape: (1, 131)
------------------------------
Field: 'stimuli_mats'
  - Type:  <class 'numpy.ndarray'>
  - Shape: (1, 131)
------------------------------
Field: 'speaking_mats'
  - Type:  <class 'numpy.ndarray'>
  - Shape: (1, 131)
------------------------------

=== KARA ONE DATA STRUCTURE ===
Matrix type: <class 'numpy.ndarray'>
Matrix shape: (1, 1)
Number of dimensions: 2

Interpretation (2D Matrix):
  - Axis 0 (Size 1): Check if this matches your total channel count or total samples.
  - Axis 1 (Size 1): The corresponding pairs.


In [21]:
# Check EEGLAB event triggers
# To inspect a specific file without editing the notebook, set KARA_ONE_SET_FILE.
configured_set_file = os.environ.get("KARA_ONE_SET_FILE")

if configured_set_file:
    file_path = Path(configured_set_file).expanduser().resolve()
    if not file_path.is_file():
        raise FileNotFoundError(f"Configured .set file not found: {file_path}")
else:
    set_files = sorted(DATASET_ROOT.rglob("*.set"))
    if not set_files:
        raise FileNotFoundError(
            f"No EEGLAB .set files were found under {DATASET_ROOT}"
        )
    file_path = set_files[0]

print(f"Opening EEGLAB file: {file_path}")

raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

print("\n=== EEGLAB FILE METADATA ===")
print("Channels discovered:", len(raw.ch_names))
print("Sampling Rate:", raw.info["sfreq"], "Hz")

try:
    events, event_id = mne.events_from_annotations(raw, verbose=False)

    print(f"\nTotal event triggers found in recording: {len(events)}")
    print("\nDiscovered Event Mapping (ID to Label):")
    for label, trigger_id in event_id.items():
        print(f"  Trigger ID {trigger_id:3} ➡️ Label: '{label}'")

except Exception as e:
    print(f"Could not parse annotations: {e}")


Opening EEGLAB file...


/var/folders/tz/23bw534d5gj78qmcq71x6czh0000gn/T/ipykernel_40043/4287828864.py:6: RuntimeWarning: Estimated head radius (12.5 cm) is above the 99th percentile for adult head size. Check if the montage_units argument is correct (the default is "mm", but your channel positions may be in different units).
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)



=== EEGLAB FILE METADATA ===
Channels discovered: 62
Sampling Rate: 1000.0 Hz

Total event triggers found in recording: 0

Discovered Event Mapping (ID to Label):
